<div dir="rtl">

# اليوم الثاني — من البيانات الخام إلى أول نموذج
كل خلية تنفذ فكرة واحدة. شغّلها ثم اقرأ الناتج قبل الانتقال.

</div>

<div dir="rtl">

## قبل البدء: افتح جلسة نظيفة
شغّل الخلية التالية مرة واحدة لمسح أي متغيرات بقيت في ذاكرة Colab من تشغيل سابق. بعدها شغّل الخلايا بالترتيب من الأعلى إلى الأسفل.

</div>

In [ ]:
%reset -f
print("تم بدء جلسة نظيفة ✅")

<div dir="rtl">

## المحطة 0 — تحميل بيانات منافذ

</div>

<div dir="rtl">

### 0.1 استيراد pandas
مكتبة `pandas` هي الأداة التي سنستخدمها لقراءة الجدول وفحصه وتعديله. الاختصار `pd` مجرد اسم قصير متعارف عليه. الناتج المتوقع: لا يظهر جدول؛ فقط تصبح المكتبة جاهزة للاستخدام.

</div>

In [ ]:
import pandas as pd

<div dir="rtl">

### 0.2 تحديد رابط الملف
نحفظ عنوان ملف بيانات منافذ في متغير اسمه `DATA_URL`. هذه الخلية لا تقرأ البيانات بعد؛ بل تعرّف مكانها فقط حتى نستخدمه في الخلية التالية.

</div>

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/hanenalmayouf/FAML/main/public/data/manafeth_customers.csv'

<div dir="rtl">

### 0.3 قراءة الملف
تقرأ `read_csv()` الملف من الرابط وتحوله إلى جدول اسمه `df` من نوع DataFrame. إذا ظهرت رسالة **تم التحميل** فهذا يعني أن الجدول أصبح موجودًا في الذاكرة ويمكن فحصه.

</div>

In [ ]:
df = pd.read_csv(DATA_URL)
print('تم التحميل ✅')

<div dir="rtl">

# المحطة 1 — افهم الجدول

</div>

<div dir="rtl">

### 1.1 حجم الجدول
تعرض `shape` رقمين: عدد الصفوف أولًا ثم عدد الأعمدة. مثال: `(10000, 19)` يعني 10,000 سجل و19 عمودًا. هذه الخطوة لا تغيّر البيانات.

</div>

In [ ]:
print('عدد الصفوف والأعمدة:', df.shape)

<div dir="rtl">

### 1.2 أول خمس حالات
تعرض `head()` عينة سريعة من بداية الجدول. نستخدمها لفهم شكل الصف الواحد، وأسماء الأعمدة، ونوع القيم قبل كتابة أي تنظيف. لا تعني أن الفحص اقتصر على خمسة صفوف.

</div>

In [ ]:
display(df.head())

<div dir="rtl">

### 1.3 أسماء الأعمدة
نطبع أسماء الأعمدة كاملة حتى نعرف المعلومات المتاحة ونلاحظ أي اسم غير واضح. لاحقًا سنحدد منها الخصائص `X` والهدف `y` والأعمدة التي يجب استبعادها.

</div>

In [ ]:
print(df.columns.tolist())

<div dir="rtl">

### 1.4 أنواع الأعمدة
يعرض `dtypes` كيف قرأت pandas كل عمود: رقمًا، نصًا، أو نوعًا آخر. يهمنا ذلك لأن الأعمدة الرقمية والفئوية تحتاج طرق تجهيز مختلفة قبل دخول النموذج.

</div>

In [ ]:
display(df.dtypes.to_frame(name="نوع البيانات"))

<div dir="rtl">

**ماذا تعلمنا؟** عرفنا حجم الجدول وأسماء الأعمدة وأنواعها قبل اتخاذ أي قرار تنظيف.

</div>

<div dir="rtl">

# المحطة 2 — افحص الجودة ونظّف نسخة من البيانات

</div>

<div dir="rtl">

### 2.1 حساب القيم المفقودة
تفحص `isna()` كل خلية، ثم تجمع `sum()` عدد الخلايا الفارغة في كل عمود. نعرض الأعمدة التي تحتوي على فقد فقط حتى تكون النتيجة قصيرة وواضحة.

</div>

In [ ]:
missing_count = df.isna().sum()
display(missing_count[missing_count > 0].to_frame('عدد المفقود'))

<div dir="rtl">

### 2.2 حساب نسبة القيم المفقودة
العدد وحده قد يكون مضللًا؛ لذلك نحسب نسبة المفقود من حجم العمود. مثلًا `31.0` تعني أن 31% من قيم ذلك العمود مفقودة، وليست 31 خلية.

</div>

In [ ]:
missing_percent = df.isna().mean().mul(100).round(1)
display(missing_percent[missing_percent > 0].to_frame('نسبة المفقود %'))

<div dir="rtl">

### 2.3 إنشاء نسخة للتنظيف
نحافظ على `df` كنسخة خام للرجوع إليها، ونجري التعديلات على `clean_df`. الرمز `copy()` ينشئ نسخة مستقلة؛ لذلك لن تغيّر خطواتنا الجدول الأصلي.

</div>

In [ ]:
clean_df = df.copy()

<div dir="rtl">

### 2.4 إنشاء مؤشر لغياب التقييم
قبل تعويض `avg_rating` نحفظ معلومة أن القيمة كانت مفقودة في عمود جديد. القيمة `1` تعني أن التقييم كان مفقودًا، و`0` تعني أنه كان موجودًا. قد يكون الغياب نفسه مفيدًا للنموذج.

</div>

In [ ]:
clean_df['rating_missing'] = clean_df['avg_rating'].isna().astype(int)
display(clean_df[['avg_rating', 'rating_missing']].head())

<div dir="rtl">

### 2.5 تعويض التقييم بالوسيط
نحسب الوسيط (Median)، أي القيمة الواقعة في منتصف التقييمات، ثم نضعه مكان القيم المفقودة. اخترناه لأنه أقل تأثرًا بالقيم العالية أو المنخفضة جدًا من المتوسط. الناتج يطبع القيمة المستخدمة في التعويض.

</div>

In [ ]:
rating_median = clean_df['avg_rating'].median()
clean_df['avg_rating'] = clean_df['avg_rating'].fillna(rating_median)
print('الوسيط المستخدم:', rating_median)

<div dir="rtl">

### 2.6 تعويض العرض المفقود بفئة واضحة
`last_promo_used` عمود فئوي نصي، لذلك لا نعوضه برقم. نستخدم `never_used` لتصبح القيمة المفقودة فئة واضحة تعني أنه لا يوجد عرض مسجل.

</div>

In [ ]:
clean_df['last_promo_used'] = clean_df['last_promo_used'].fillna('never_used')

<div dir="rtl">

### 2.7 فحص الصفوف المكررة
تتحقق `duplicated()` هل يوجد صف مطابق تمامًا لصف سابق. نحن نحسب التكرار أولًا ولا نحذفه تلقائيًا؛ لأن السجل المتشابه قد يكون حدثًا حقيقيًا وليس نسخة خاطئة.

</div>

In [ ]:
duplicate_rows = clean_df.duplicated().sum()
print('عدد الصفوف المكررة:', duplicate_rows)

<div dir="rtl">

### 2.8 فحص نطاق التقييم
بحسب تعريف العمود يجب أن يكون التقييم بين 1 و5. تتحقق `between(1, 5)` من هذا الشرط، والرمز `~` يعكس النتيجة ليحدد القيم الخارجة عن النطاق.

</div>

In [ ]:
invalid_rating = ~clean_df['avg_rating'].between(1, 5)
print('عدد التقييمات خارج 1 إلى 5:', invalid_rating.sum())

<div dir="rtl">

### 2.9 مشاهدة أعلى قيم السلة قبل اتخاذ القرار
تعرض `nlargest()` أكبر عشر قيم في `avg_basket_sar`. الهدف فحص القيم المرتفعة مع رقم العميل، لا حذفها مباشرة؛ فقد تكون عملية شراء كبيرة وصحيحة.

</div>

In [ ]:
display(clean_df.nlargest(10, 'avg_basket_sar')[['customer_id', 'avg_basket_sar']])

<div dir="rtl">

**القرار المحلول:** نبقي القيمة المرتفعة؛ لأنها ممكنة وليست دليلًا كافيًا على خطأ إدخال.

</div>

<div dir="rtl">

# المحطة 3 — جهّز X وy ومنع التسريب

</div>

<div dir="rtl">

### 3.1 حفظ الهدف y أولًا
الهدف `y` هو الإجابة التي نريد من النموذج تعلمها: هل توقف العميل خلال 30 يومًا؟ ننسخ عمود `churned_30d` قبل حذفه من الخصائص حتى يبقى منفصلًا وواضحًا.

</div>

In [ ]:
y = clean_df['churned_30d'].copy()

<div dir="rtl">

### 3.2 تحديد الأعمدة التي لا تدخل في X
نكتب قائمة الأعمدة المستبعدة لسببين: أعمدة تعريفية مثل `customer_id`، وأعمدة تكشف المستقبل مثل `next_month_orders`. وجود الهدف نفسه أو معلومة ظهرت بعد زمن التنبؤ داخل `X` يسبب تسريب بيانات (Data Leakage).

</div>

In [ ]:
drop_cols = [
    'customer_id', 'churned_30d',
    'refund_issued', 'support_ticket_after_snapshot', 'next_month_orders',
    'signup_date', 'snapshot_date'
]

<div dir="rtl">

### 3.3 إنشاء X
الخصائص `X` هي المعلومات التي سيراها النموذج عند إصدار التوقع. تنشئ `drop()` جدولًا جديدًا بعد استبعاد الأعمدة الموجودة في `drop_cols`، بينما يبقى `y` محفوظًا منفصلًا.

</div>

In [ ]:
X = clean_df.drop(columns=drop_cols, errors="ignore")

<div dir="rtl">

### 3.4 التأكد من الحجم
يجب أن يكون عدد صفوف `X` مساويًا لعدد قيم `y`؛ لأن كل صف عميل يجب أن يقابله هدف واحد. اختلاف العددين يعني أن شيئًا حدث أثناء التجهيز ويجب مراجعته.

</div>

In [ ]:
print('X:', X.shape)
print('y:', y.shape)

<div dir="rtl">

# المحطة 4 — تقسيم البيانات وبناء Pipeline

</div>

<div dir="rtl">

### 4.1 تقسيم البيانات
نقسم الصفوف إلى 80% للتدريب و20% للاختبار. يتعلم النموذج من `X_train` و`y_train` فقط، ثم نستخدم `X_test` و`y_test` لقياس أدائه على حالات لم يرها. يحافظ `stratify=y` على نسبة الفئتين، ويجعل `random_state=42` التقسيم قابلًا للتكرار.

</div>

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

<div dir="rtl">

### 4.2 معرفة الأعمدة الرقمية
نستخرج أسماء الأعمدة الرقمية تلقائيًا، مثل عدد الطلبات ومتوسط السلة. سنرسل هذه الأعمدة إلى مسار يعوض المفقود ثم يطبق Scaling.

</div>

In [ ]:
numeric_cols = X.select_dtypes(include='number').columns.tolist()
print(numeric_cols)

<div dir="rtl">

### 4.3 معرفة الأعمدة الفئوية
نستخرج الأعمدة غير الرقمية، مثل المدينة ونوع العرض. لا يستطيع النموذج استخدام النص مباشرة، لذلك سنرسلها إلى مسار يعوض المفقود ثم يطبق Encoding.

</div>

In [ ]:
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()
print(categorical_cols)

<div dir="rtl">

### 4.4 مسار تجهيز الأرقام
يجمع `numeric_pipe` خطوتين بالترتيب: يعوض أي رقم مفقود بالوسيط، ثم يستخدم `StandardScaler` لتقريب مقاييس الأعمدة من بعضها. لا نشغله يدويًا الآن؛ سيعمل داخل النموذج عند `fit`.

</div>

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

<div dir="rtl">

### 4.5 مسار تجهيز الفئات
يجمع `categorical_pipe` خطوتين: يعوض النص المفقود بأكثر فئة تكرارًا، ثم يحول الفئات إلى أعمدة رقمية باستخدام `OneHotEncoder`. يسمح `handle_unknown='ignore'` بظهور فئة جديدة في الاختبار دون تعطل الكود.

</div>

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore'))
])

<div dir="rtl">

### 4.6 جمع المسارين
يطبّق `ColumnTransformer` المسار الرقمي على `numeric_cols` والمسار الفئوي على `categorical_cols` في الوقت نفسه، ثم يجمع الناتجين في جدول واحد جاهز للنموذج. هذا الكائن اسمه `preprocess`.

</div>

In [ ]:
from sklearn.compose import ColumnTransformer

preprocess = ColumnTransformer([
    ('numbers', numeric_pipe, numeric_cols),
    ('categories', categorical_pipe, categorical_cols)
])

<div dir="rtl">

# المحطة 5 — Baseline ثم نموذج تصنيف

</div>

<div dir="rtl">

### 5.1 إنشاء خط الأساس (Baseline)
خط الأساس مرجع بسيط جدًا: `DummyClassifier` يتوقع دائمًا الفئة الأكثر ظهورًا ولا يتعلم علاقات حقيقية. نستخدمه لمعرفة هل نموذجنا الفعلي أضاف قيمة أم لا.

</div>

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = Pipeline([
    ('preprocess', preprocess),
    ('model', DummyClassifier(strategy='most_frequent'))
])

<div dir="rtl">

### 5.2 تدريب خط الأساس
تشغّل `fit()` خطوات التجهيز على بيانات التدريب، ثم يتعرف خط الأساس على الفئة الأكثر تكرارًا في `y_train`. لا تظهر نتيجة مرئية؛ نجاح الخلية يعني أن المرجع أصبح جاهزًا.

</div>

In [ ]:
baseline.fit(X_train, y_train)

<div dir="rtl">

### 5.3 قياس Accuracy لخط الأساس
تقيس `score()` نسبة توقعات خط الأساس الصحيحة على بيانات الاختبار. نحفظها في `baseline_accuracy` لنقارن بها النموذج الحقيقي لاحقًا؛ الرقم المرتفع لا يعني وحده أن الخط الأساسي ذكي.

</div>

In [ ]:
baseline_accuracy = baseline.score(X_test, y_test)
print('Baseline Accuracy:', round(baseline_accuracy, 3))

<div dir="rtl">

### 5.4 إنشاء Logistic Regression
نبني Pipeline جديدة تبدأ بالتجهيز نفسه، ثم تستخدم `LogisticRegression` للتصنيف. النموذج سيتعلم من الخصائص احتمال أن تكون الفئة `1`، أي احتمال توقف العميل.

</div>

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=1000))
])

<div dir="rtl">

### 5.5 تدريب النموذج
عند `fit()` يتعلم التجهيز قواعده من `X_train` فقط، ثم يتعلم Logistic Regression أوزان الخصائص بمقارنة توقعاته مع `y_train`. لا نكتب الأوزان يدويًا؛ التدريب هو الذي يحددها.

</div>

In [ ]:
logistic_model.fit(X_train, y_train)

<div dir="rtl">

### 5.6 إنتاج الفئات المتوقعة
تستخدم `predict()` النموذج المدرّب لإعطاء قرار نهائي لكل صف في `X_test`: القيمة `1` تعني متوقع أن يتوقف، و`0` تعني متوقع ألا يتوقف. نحفظ جميع القرارات في `test_prediction`.

</div>

In [ ]:
test_prediction = logistic_model.predict(X_test)

<div dir="rtl">

### 5.7 إنتاج احتمال الفئة 1
تعيد `predict_proba()` احتمال كل فئة. نختار العمود `[:, 1]` لأنه يمثل احتمال الفئة `1`، أي التوقف. الاحتمال يعطي تفاصيل أكثر من القرار النهائي؛ فقد يكون 0.51 أو 0.95 مع أن كليهما يصبح توقعه `1`.

</div>

In [ ]:
test_probability = logistic_model.predict_proba(X_test)[:, 1]

<div dir="rtl">

### 5.8 مشاهدة أول عشر نتائج
نجمع الحقيقة والتوقع والاحتمال في جدول واحد لأول عشر حالات فقط كي نتعلم قراءة النتيجة. هذه عينة للشرح وليست تقييمًا كاملًا؛ القياس الحقيقي سيستخدم جميع صفوف الاختبار في الخطوة التالية.

</div>

In [ ]:
result = pd.DataFrame({
    'الحقيقة': y_test.iloc[:10].values,
    'التوقع': test_prediction[:10],
    'احتمال التوقف': test_probability[:10].round(3)
})
display(result)

<div dir="rtl">

### كيف نقرأ الجدول السابق؟

- **الحقيقة:** ما حدث فعلًا للعميل: `1` توقف، و`0` لم يتوقف.
- **احتمال التوقف:** تقدير النموذج قبل اتخاذ القرار. مثال: `0.78` يعني أن النموذج يقدّر احتمال التوقف بـ78%.
- **التوقع:** القرار الناتج من الاحتمال. افتراضيًا، إذا كان الاحتمال `0.50` أو أكثر يصبح التوقع `1`، وإلا يصبح `0`.
- إذا اختلفت **الحقيقة** عن **التوقع** فهذا خطأ من النموذج، وهو أمر طبيعي. لا نحكم من عشرة صفوف فقط؛ نقيس الأداء على جميع صفوف الاختبار.

لا نعدّل أوزان النموذج يدويًا بعد رؤية الصفوف المختلفة. النموذج تعلّمها أثناء `fit`. لاحقًا يمكن مقارنة نموذج آخر أو تغيير عتبة القرار، لكن أولًا نقيس أداء النموذج الحالي.

</div>

<div dir="rtl">

### 5.9 حساب الدقة (Accuracy)

الدقة هي نسبة التوقعات الصحيحة من جميع حالات الاختبار. إذا كانت الدقة `0.82` فهذا يعني أن النموذج أعطى توقعًا صحيحًا في 82 من كل 100 حالة تقريبًا.

</div>

In [ ]:
from sklearn.metrics import accuracy_score

model_accuracy = accuracy_score(y_test, test_prediction)
print('Model Accuracy:', round(model_accuracy, 3))

<div dir="rtl">

### 5.10 مقارنة النموذج بخط الأساس (Baseline)

لا تكفي الدقة وحدها؛ نتحقق أولًا أن النموذج أفضل من الحل البسيط الذي يتوقع دائمًا الفئة الأكثر شيوعًا.

</div>

In [ ]:
print('Baseline Accuracy:', round(baseline_accuracy, 3))
print('Model Accuracy:   ', round(model_accuracy, 3))
print('Improvement:      ', round(model_accuracy - baseline_accuracy, 3))

<div dir="rtl">

### 5.11 أين أخطأ النموذج؟

يعرض الجدول التالي عدد الحالات بدل الاكتفاء بنسبة واحدة. الصفوف هي الحقيقة، والأعمدة هي توقع النموذج. القيم على القطر صحيحة، والقيم خارجه أخطاء.

</div>

In [ ]:
comparison_table = pd.crosstab(
    y_test,
    test_prediction,
    rownames=['الحقيقة'],
    colnames=['التوقع'],
    margins=True
)
display(comparison_table)

<div dir="rtl">

هذه أول قراءة لأداء نموذج التصنيف. في اليوم الثالث سنعطي الأخطاء أسماءها الأكاديمية، ثم نتعلم لماذا لا تكفي Accuracy وحدها ومتى نستخدم Precision وRecall.

</div>

<div dir="rtl">

## الخلاصة
مررنا بالترتيب: فهم الجدول ← فحص الجودة ← التنظيف ← تحديد X وy ← التقسيم ← تجهيز البيانات ← Baseline ← أول نموذج ← مقارنة الحقيقة بالتوقع ← قياس Accuracy.

</div>